# 医用画像AI異常検知 実習ノートブック（全12回）

対象：大学3年生程度  
環境：Google Colab + Google Drive  
題材：PneumoniaMNIST（小児胸部X線画像の正常／肺炎データ）  
中心課題：**正常画像だけでAutoencoderを学習し，再構成誤差で異常検知する**

このノートブックは，12回の授業で少しずつ進められるように作ってあります。各回の到達目標・確認事項・発展課題をMarkdownセルに入れ，すぐ下に実行可能なコードを配置しています。


## 全12回の進行案

| 回 | 主題 | 到達目標 |
|---:|---|---|
| 1 | ColabとGoogle Driveの準備 | ノートブック，GPU，Drive保存先を理解する |
| 2 | 医用画像データセットの取得 | PneumoniaMNISTをダウンロードし，画像・ラベルを確認する |
| 3 | Python / NumPy / Tensorの基礎 | 配列，shape，正規化，ミニバッチを理解する |
| 4 | PyTorch Dataset / DataLoader | 学習用・検証用・テスト用データの流れを理解する |
| 5 | 異常検知の考え方 | 「正常だけを学習する」異常検知の発想を理解する |
| 6 | Autoencoderの構築 | Encoder/Decoder，潜在表現，再構成を理解する |
| 7 | 学習ループ | loss，optimizer，epoch，GPU実行を理解する |
| 8 | 再構成誤差による異常スコア | 正常と異常のスコア分布を比較する |
| 9 | 評価指標 | ROC-AUC，F1，混同行列，しきい値を理解する |
| 10 | 可視化と説明 | 再構成画像・誤差ヒートマップを見る |
| 11 | モデル保存と再利用 | Driveにモデル・結果を保存し，読み直す |
| 12 | 発展：教師ありCNNとの比較 | 異常検知と分類の違い，研究課題化の方向を考える |


## 第1回：ColabとGoogle Driveの準備

最初に，Google Driveをマウントし，実習で使うファイルを永続保存できるようにします。

Colabのメニューで `ランタイム > ランタイムのタイプを変更` から GPU を選んでください。CPUでも動きますが，GPUの方が快適です。


In [ ]:
# 第1回：Google Driveをマウントし，作業フォルダを作る
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/medical_ai_anomaly_practice')
DATA_DIR = PROJECT_DIR / 'data'
MODEL_DIR = PROJECT_DIR / 'models'
RESULT_DIR = PROJECT_DIR / 'results'

for d in [PROJECT_DIR, DATA_DIR, MODEL_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('作業フォルダ:', PROJECT_DIR)
print('データ保存先:', DATA_DIR)
print('モデル保存先:', MODEL_DIR)
print('結果保存先:', RESULT_DIR)


## 第2回：ライブラリのインストール

MedMNISTは，教育・研究用途に扱いやすい標準化済み医用画像データセットです。ここでは `medmnist` パッケージから PneumoniaMNIST を取得します。


In [ ]:
# 第2回：必要なライブラリをインストール
!pip -q install medmnist scikit-learn tqdm pandas matplotlib

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, Dataset

from torchvision import transforms

from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_fscore_support,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

from tqdm.auto import tqdm

import medmnist
from medmnist import INFO

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## 第3回：乱数固定と実行デバイス

AI実験では，乱数によって結果が少し変わります。授業では比較しやすいように乱数を固定します。


In [ ]:
# 第3回：乱数固定とGPU/CPUの確認
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


## 第4回：PneumoniaMNISTのダウンロードと確認

PneumoniaMNISTは，胸部X線画像を正常／肺炎に分けた2値分類データです。  
この実習では，分類問題として直接解く前に，**正常画像だけを見せてAIに正常らしさを学習させる**方法を扱います。


In [ ]:
# 第4回：PneumoniaMNISTをダウンロードして読み込む
DATA_FLAG = 'pneumoniamnist'
info = INFO[DATA_FLAG]
DataClass = getattr(medmnist, info['python_class'])

print('Dataset:', DATA_FLAG)
print('Task:', info['task'])
print('Channels:', info['n_channels'])
print('Labels:', info['label'])

transform = transforms.Compose([
    transforms.ToTensor(),  # 0〜1のTensorに変換，shape: [C, H, W]
])

train_dataset = DataClass(split='train', transform=transform, download=True, root=str(DATA_DIR))
val_dataset   = DataClass(split='val',   transform=transform, download=True, root=str(DATA_DIR))
test_dataset  = DataClass(split='test',  transform=transform, download=True, root=str(DATA_DIR))

print('train:', len(train_dataset))
print('val  :', len(val_dataset))
print('test :', len(test_dataset))

x0, y0 = train_dataset[0]
print('image shape:', x0.shape)
print('label shape:', y0.shape, 'label:', y0)


## 第5回：画像とラベルを可視化する

まずはデータを「見る」ことが重要です。  
AIに入力する前に，画像サイズ，濃淡，ラベルの意味を確認します。


In [ ]:
# 第5回：画像サンプルを表示
label_names = info['label']  # 例: {'0': 'normal', '1': 'pneumonia'}

def show_samples(dataset, n=12):
    plt.figure(figsize=(12, 4))
    indices = np.random.choice(len(dataset), size=n, replace=False)
    for i, idx in enumerate(indices):
        img, label = dataset[idx]
        label_int = int(np.array(label).reshape(-1)[0])
        plt.subplot(2, n//2, i+1)
        plt.imshow(img.squeeze(), cmap='gray')
        plt.title(f'{label_int}: {label_names[str(label_int)]}')
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show_samples(train_dataset, n=12)


## 第6回：ラベル分布を確認する

異常検知では，正常データと異常データの数の偏りが重要です。  
ここでは，正常を `0`，異常を `1` として扱います。


In [ ]:
# 第6回：ラベル分布を確認
def labels_to_numpy(dataset):
    labels = []
    for _, y in dataset:
        labels.append(int(np.array(y).reshape(-1)[0]))
    return np.array(labels)

train_labels = labels_to_numpy(train_dataset)
val_labels   = labels_to_numpy(val_dataset)
test_labels  = labels_to_numpy(test_dataset)

def print_label_count(name, labels):
    unique, counts = np.unique(labels, return_counts=True)
    print(name)
    for u, c in zip(unique, counts):
        print(f'  {u}: {label_names[str(int(u))]} -> {c}')

print_label_count('train', train_labels)
print_label_count('val', val_labels)
print_label_count('test', test_labels)


## 第7回：異常検知用データセットを作る

Autoencoderによる異常検知では，学習時には正常画像だけを使います。  
検証・テスト時には，正常と異常の両方を使って，異常を見分けられるか確認します。


In [ ]:
# 第7回：正常画像だけの学習データを作る
NORMAL_LABEL = 0
ANOMALY_LABEL = 1

normal_train_indices = np.where(train_labels == NORMAL_LABEL)[0]
normal_train_dataset = Subset(train_dataset, normal_train_indices)

BATCH_SIZE = 128

normal_train_loader = DataLoader(
    normal_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print('正常のみの学習データ数:', len(normal_train_dataset))


## 第8回：Autoencoderを定義する

Autoencoderは，入力画像をいったん小さな潜在表現に圧縮し，そこから元画像を再構成します。

正常画像だけで学習すると，正常画像はうまく再構成できる一方，異常画像は再構成しにくくなることが期待されます。


In [ ]:
# 第8回：畳み込みAutoencoderの定義
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),  # 28 -> 14
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # 14 -> 7
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, latent_dim),
            nn.ReLU()
        )

        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 32 * 7 * 7),
            nn.ReLU()
        )

        self.decoder_conv = nn.Sequential(
            nn.Unflatten(1, (32, 7, 7)),
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1), # 7 -> 14
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),  # 14 -> 28
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        h = self.decoder_fc(z)
        out = self.decoder_conv(h)
        return out

model = ConvAutoencoder(latent_dim=32).to(device)
print(model)


## 第9回：学習ループを作る

ここでAI開発の基本である，順伝播，損失計算，誤差逆伝播，パラメータ更新を実行します。  
損失関数は，入力画像と再構成画像の平均二乗誤差（MSE）です。


In [ ]:
# 第9回：Autoencoderを学習する
def train_autoencoder(model, train_loader, epochs=10, lr=1e-3):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for x, _ in tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False):
            x = x.to(device)

            optimizer.zero_grad()
            recon = model(x)
            loss = criterion(recon, x)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)

        avg_loss = total_loss / len(train_loader.dataset)
        history.append(avg_loss)
        print(f'Epoch {epoch:02d} | train loss = {avg_loss:.6f}')

    return history

EPOCHS = 10
history = train_autoencoder(model, normal_train_loader, epochs=EPOCHS, lr=1e-3)

plt.figure(figsize=(6, 4))
plt.plot(history, marker='o')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Autoencoder Training Loss')
plt.grid(True)
plt.show()


## 第10回：再構成誤差を異常スコアにする

画像ごとに，入力画像と再構成画像の差を計算します。  
この値が大きいほど，モデルにとって「見慣れない画像」，すなわち異常らしい画像とみなします。


In [ ]:
# 第10回：再構成誤差を計算する
@torch.no_grad()
def compute_anomaly_scores(model, loader):
    model.eval()
    scores = []
    labels = []

    for x, y in loader:
        x = x.to(device)
        recon = model(x)

        # 画像ごとの平均二乗誤差
        batch_scores = torch.mean((x - recon) ** 2, dim=(1, 2, 3))

        scores.extend(batch_scores.cpu().numpy())
        labels.extend(y.numpy().reshape(-1).astype(int))

    return np.array(scores), np.array(labels)

val_scores, val_y = compute_anomaly_scores(model, val_loader)
test_scores, test_y = compute_anomaly_scores(model, test_loader)

print('val_scores:', val_scores.shape)
print('test_scores:', test_scores.shape)

plt.figure(figsize=(7, 4))
plt.hist(val_scores[val_y == NORMAL_LABEL], bins=30, alpha=0.6, label='normal')
plt.hist(val_scores[val_y == ANOMALY_LABEL], bins=30, alpha=0.6, label='pneumonia/anomaly')
plt.xlabel('Reconstruction error')
plt.ylabel('Count')
plt.title('Validation anomaly score distribution')
plt.legend()
plt.grid(True)
plt.show()


## 第11回：しきい値を決めて評価する

異常スコアが高いほど異常とみなします。  
検証データでF1スコアが最大になるしきい値を選び，テストデータで性能を評価します。


In [ ]:
# 第11回：しきい値選択と評価
def find_best_threshold(scores, labels):
    thresholds = np.linspace(scores.min(), scores.max(), 200)
    best = {'threshold': None, 'f1': -1, 'precision': None, 'recall': None}

    for th in thresholds:
        pred = (scores >= th).astype(int)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, pred, average='binary', zero_division=0
        )
        if f1 > best['f1']:
            best.update({
                'threshold': th,
                'f1': f1,
                'precision': precision,
                'recall': recall
            })
    return best

best = find_best_threshold(val_scores, val_y)
print('Best threshold on validation:')
print(best)

test_pred = (test_scores >= best['threshold']).astype(int)

auc = roc_auc_score(test_y, test_scores)
precision, recall, f1, _ = precision_recall_fscore_support(
    test_y, test_pred, average='binary', zero_division=0
)

print(f'Test ROC-AUC : {auc:.4f}')
print(f'Test Precision: {precision:.4f}')
print(f'Test Recall   : {recall:.4f}')
print(f'Test F1       : {f1:.4f}')

print('\nClassification report:')
print(classification_report(test_y, test_pred, target_names=['normal', 'anomaly'], zero_division=0))

cm = confusion_matrix(test_y, test_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['normal', 'anomaly'])
disp.plot(values_format='d')
plt.title('Confusion Matrix')
plt.show()

fpr, tpr, _ = roc_curve(test_y, test_scores)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f'AUC={auc:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.show()


## 第12回：再構成画像と誤差ヒートマップを可視化する

異常検知では，数値だけでなく「どこが違って見えているか」を確認することが重要です。  
ここでは，入力画像，再構成画像，差分画像を並べて表示します。


In [ ]:
# 第12回：再構成画像と差分ヒートマップ
@torch.no_grad()
def visualize_reconstruction(model, dataset, indices=None, n=8):
    model.eval()

    if indices is None:
        indices = np.random.choice(len(dataset), size=n, replace=False)

    plt.figure(figsize=(12, 3 * n))

    for row, idx in enumerate(indices):
        x, y = dataset[idx]
        label_int = int(np.array(y).reshape(-1)[0])

        x_batch = x.unsqueeze(0).to(device)
        recon = model(x_batch).cpu().squeeze(0)

        diff = torch.abs(x - recon)

        imgs = [x.squeeze(), recon.squeeze(), diff.squeeze()]
        titles = [
            f'Input: {label_names[str(label_int)]}',
            'Reconstruction',
            'Absolute Error'
        ]

        for col in range(3):
            plt.subplot(n, 3, row * 3 + col + 1)
            plt.imshow(imgs[col], cmap='gray')
            plt.title(titles[col])
            plt.axis('off')

    plt.tight_layout()
    plt.show()

# 正常例と異常例を混ぜて表示
normal_test_indices = np.where(test_y == NORMAL_LABEL)[0][:4]
anomaly_test_indices = np.where(test_y == ANOMALY_LABEL)[0][:4]
indices = np.concatenate([normal_test_indices, anomaly_test_indices])
visualize_reconstruction(model, test_dataset, indices=indices, n=len(indices))


## 補足A：モデルと結果をGoogle Driveに保存する

Colabのランタイムは切れることがあります。学習済みモデルや評価結果はDriveに保存します。


In [ ]:
# 補足A：モデルと結果を保存
model_path = MODEL_DIR / 'pneumonia_autoencoder.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'data_flag': DATA_FLAG,
    'normal_label': NORMAL_LABEL,
    'anomaly_label': ANOMALY_LABEL,
    'threshold': float(best['threshold']),
    'history': history,
}, model_path)

result_df = pd.DataFrame({
    'score': test_scores,
    'label': test_y,
    'pred': test_pred
})
result_path = RESULT_DIR / 'test_anomaly_scores.csv'
result_df.to_csv(result_path, index=False)

print('Saved model to:', model_path)
print('Saved results to:', result_path)


## 補足B：保存したモデルを読み込む

次回以降の授業では，学習済みモデルを読み込んで評価や可視化から再開できます。


In [ ]:
# 補足B：モデルを読み込む例
loaded = ConvAutoencoder(latent_dim=32).to(device)
checkpoint = torch.load(model_path, map_location=device)
loaded.load_state_dict(checkpoint['model_state_dict'])
loaded.eval()

loaded_threshold = checkpoint['threshold']
print('Loaded threshold:', loaded_threshold)


## 補足C：教師ありCNNとの比較

最後に，正常・肺炎ラベルを使って普通の分類器も作ります。  
Autoencoder型異常検知との違いを考察してください。

- 教師あり分類：正常・異常の両方のラベルが必要
- 異常検知：正常データだけでも学習可能
- 医療応用では，異常の種類が多く，未知異常が出ることがある


In [ ]:
# 補足C：簡単な教師ありCNN分類器
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 28 -> 14
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 14 -> 7
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)

train_loader_full = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

cnn = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)

def train_classifier(model, loader, epochs=5):
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for x, y in tqdm(loader, desc=f'CNN Epoch {epoch}/{epochs}', leave=False):
            x = x.to(device)
            y = y.view(-1).long().to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += x.size(0)

        print(f'Epoch {epoch:02d} | loss={total_loss/total:.4f} | acc={correct/total:.4f}')

train_classifier(cnn, train_loader_full, epochs=5)

@torch.no_grad()
def evaluate_classifier(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    y_prob = []

    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        prob = torch.softmax(logits, dim=1)[:, 1]
        pred = logits.argmax(dim=1)

        y_true.extend(y.numpy().reshape(-1).astype(int))
        y_pred.extend(pred.cpu().numpy())
        y_prob.extend(prob.cpu().numpy())

    return np.array(y_true), np.array(y_pred), np.array(y_prob)

cnn_y, cnn_pred, cnn_prob = evaluate_classifier(cnn, test_loader)
cnn_auc = roc_auc_score(cnn_y, cnn_prob)

print('CNN Test ROC-AUC:', cnn_auc)
print(classification_report(cnn_y, cnn_pred, target_names=['normal', 'pneumonia'], zero_division=0))


## 発展課題

1. Autoencoderの潜在次元 `latent_dim` を 8, 16, 32, 64 に変えて結果を比較する。
2. Epoch数を増やすと異常検知性能は上がるか，下がるかを確認する。
3. しきい値をF1最大ではなく，Recall重視で決めるとどうなるかを調べる。
4. 誤検出された画像を可視化し，なぜ間違えたかを考察する。
5. PneumoniaMNIST以外のMedMNISTデータセットに差し替えて実験する。
6. 教師ありCNNとAutoencoder異常検知の長所・短所を表にまとめる。
7. 医療AIとして実運用する場合に必要な倫理・安全性・説明責任を議論する。
